# 不计分实验：逻辑回归中的过拟合。

讲课中描述了**过拟合**，即模型过于贴近数据、泛化不好。在本实验中，我们将探索逻辑回归中的过拟合，以及正则化如何改善这种状况。


## 目标
在本实验中，你将：
- 使用 `map_features` 扩展数据集的特征
- 探索由此产生的过拟合
- 利用正则化减轻过拟合
- 减少特征以匹配数据并减轻过拟合。

# 目录
- [工具](#tools)
- [数据集](#dataset)
- [多项式特征映射](#FeatureMap)
- [拟合模型](#FitModel)
- [减轻过拟合](#ReduceOverfitting)

# 过拟合
在本实验中，我们将探索过拟合如何发生，以及可以采取什么措施。
- 创建一个边界不规则的逻辑数据集
- 制造一个过拟合问题
    - 多项式回归与特征映射
- 用正则化减轻过拟合
<a name='tools'></a>
## 工具
- 我们还没有开发出带正则化的完整梯度下降能力，因此我们将利用 sklearn 的 LogisticRegression 功能，之前在某个实验中已简要介绍过。
- 在探索决策边界时，绘图非常有用。我们将使用 matplotlib。生成这些图涉及不少细节，因此下面提供了辅助程序。
- 我们将创建多项式特征集。`map_features` 就是为简化这一过程而提供的

In [24]:
import numpy as np
from IPython.display import Markdown as md
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
plt.rcParams['font.size'] = 8
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn import preprocessing
from plt_overfit import map_feature, plot_decision_boundary, plt_overfit
from lab_utils_common import dlc, plot_data, zscore_normalize_features, gradient_descent, predict_logistic, predict_linear

In [25]:
def map_one_feature(X1, degree):
    """映射到多项式特征的特征映射函数"""
    X1 = np.atleast_1d(X1)
    out = []
    str = ""
    k = 0
    for i in range(1, degree+1):
        out.append((X1**i))
        str = str + f"w_{{{k}}}{munge('x_0',i)} + "
        k += 1
    str = str + ' b' #把 b 加到文字公式中，而不是加到数据中
    return np.stack(out, axis=1), str 

def munge(base,exp):
    if exp == 0:
        return ('')
    elif exp == 1:
        return (base)
    else:
        return (base + f'^{{{exp}}}')



In [26]:
a = np.array([1.,2,3])
a_mapped, a_eq = map_one_feature(a,3)
display(md(f"${a_eq}$"))
print(a_eq)
print(a_mapped.shape)
print(a_mapped)

$w_{0}x_0 + w_{1}x_0^{2} + w_{2}x_0^{3} +  b$

w_{0}x_0 + w_{1}x_0^{2} + w_{2}x_0^{3} +  b
(3, 3)
[[ 1.  1.  1.]
 [ 2.  4.  8.]
 [ 3.  9. 27.]]


<a name='dataset'></a>
##  数据集
下面我们基于一个二次函数创建含两个特征的逻辑数据集。加入随机噪声以形成模型可能过拟合的场景。

In [27]:
m = 50
n = 2
np.random.seed(2)
X_train = 2*(np.random.rand(m,n)-[0.5,0.5])
y_train = X_train[:,1]+0.5  > X_train[:,0]**2 + 0.5*np.random.rand(m) #二次 + 随机
y_train = y_train + 0  #把布尔值转换为整数

fig, ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax, s=10, loc='lower right')
ax.set_title("Logistic data set with noise")
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [28]:
from matplotlib.gridspec import GridSpec
from matplotlib.widgets import Button, CheckButtons
from matplotlib.patches import FancyArrowPatch
import math
# 用于调试
from ipywidgets import Output




In [29]:
output = Output() # 在使用部件时把隐藏的错误信息显示出来
display(output)


Output()

In [30]:
class button_manager:
    '''处理 matplotlib 复选框的一些缺失功能
    初始化时：
        创建按钮，链接到 button_click 程序，
        以 active index 和 firsttime=True 调用 call_on_click
    点击时：
        保持单个按钮处于选中状态，调用 call_on_click'''
    @output.capture()  # 调试
    def __init__(self,fig, dim, labels, init, call_on_click):
        '''dim: (list)     [左下_x,下_y,宽,高]
        labels: (list)  例如 ['1','2','3','4','5','6']
        init: (list)    例如 [True, False, False, False, False, False]'''
        self.fig = fig
        self.ax = plt.axes(dim)  #lx,by,w,h
        self.init_state = init    
        self.call_on_click = call_on_click
        self.button  = CheckButtons(self.ax,labels,init)
        self.button.on_clicked(self.button_click)
        self.status = self.button.get_status()
        self.call_on_click(self.status.index(True),firsttime=True)
        
    @output.capture()  # 调试
    def reinit(self):
        self.status = self.init_state
        self.button.set_active(self.status.index(True))      #关闭旧的，将触发更新并设置状态
  
    @output.capture()  # 调试
    def button_click(self, event):
        '''保持单选状态。如果已选中的按钮被点击，也能正确处理'''
        new_status = self.button.get_status()
        new = [self.status[i] ^ new_status[i] for i in range(len(self.status))]
        newidx = new.index(True)
        self.button.eventson = False
        self.button.set_active(self.status.index(True))  #关闭旧的；若相同则重新启用
        self.button.eventson = True
        self.status = self.button.get_status()
        self.call_on_click(self.status.index(True))
        

In [33]:
class overfit_example():
    def __init__(self, X, y, w_in, b_in, regularize=False):
        self.X = X
        self.y = y
        self.w = w_in
        self.b = b_in
        self.regularize=regularize
        self.lambda_=0
        fig = plt.figure( figsize=(8,6))
        fig.canvas.toolbar_visible = False
        fig.canvas.header_visible = False
        fig.canvas.footer_visible = False
        fig.set_facecolor('#ffffff') #白色
        gs  = GridSpec(5, 3, figure=fig)
        ax0 = fig.add_subplot(gs[0:3, :])
        ax1 = fig.add_subplot(gs[-2, :])
        ax2 = fig.add_subplot(gs[-1, :])
        ax1.set_axis_off()
        ax2.set_axis_off()
        self.ax = [ax0,ax1,ax2]
        self.fig = fig

        #pos = ax2.get_position().get_points()  ##[[lb_x,lb_y], [rt_x, rt_y]]
        #print(pos)
        self.axfitdata = plt.axes([0.26,0.124,0.10,0.1 ])  #lx,by,w,h
        self.bfitdata  = Button(self.axfitdata , 'fit data', color=dlc['dlblue'])
        self.bfitdata.label.set_fontsize(12)
        self.bfitdata.on_clicked(self.fitdata_clicked)

        self.cid = fig.canvas.mpl_connect('button_press_event', self.add_data)

        self.typebut = button_manager(fig, [0.4, 0.07,0.15,0.15], ["Regression", "Categorical"],
                                       [False,True], self.toggle_type)

        self.fig.text(0.1, 0.02+0.21, "Degree", fontsize=12)
        self.degrbut = button_manager(fig,[0.1,0.02,0.15,0.2 ], ['1','2','3','4','5','6'], 
                                        [True, False, False, False, False, False], self.update_equation)
        if self.regularize:
            self.fig.text(0.6, 0.02+0.21, r"lambda($\lambda$)", fontsize=12)
            self.lambut = button_manager(fig,[0.6,0.02,0.15,0.2 ], ['0.0','0.2','0.4','0.6','0.8','1'], 
                                        [True, False, False, False, False, False], self.updt_lambda)
   
        #self.regbut =  button_manager(fig, [0.8, 0.08,0.24,0.15], ["Regularize"],
        #                               [False], self.toggle_reg)
        #self.logistic_data()
    
    def updt_lambda(self, idx, firsttime=False):
        self.lambda_ = idx * 0.2
        
    def toggle_type(self, idx, firsttime=False):
        self.logistic = True if idx==1 else False
        self.ax[0].clear()
        if self.logistic:
            self.logistic_data()
        else:
            self.linear_data()
        if not firsttime: self.degrbut.reinit()
        
    def logistic_data(self,redraw=False):
        if not redraw:
            m = 50
            n = 2
            np.random.seed(2)
            X_train = 2*(np.random.rand(m,n)-[0.5,0.5])
            y_train = X_train[:,1]+0.5  > X_train[:,0]**2 + 0.5*np.random.rand(m) #二次 + 随机
            y_train = y_train + 0  #把布尔值转换为整数
            self.X = X_train
            self.y = y_train 

        #plot_data(X_train, y_train, self.ax[0], s=10, loc='lower right')
        plot_data(self.X, self.y, self.ax[0], s=10, loc='lower right')
        self.ax[0].set_title("Logistic data set with noise")
        self.ax[0].text(0.5,0.93, "Click on plot to add data. Hold [Shift] for blue(y=0) data.",
                        fontsize=12, ha='center',transform=self.ax[0].transAxes, color=dlc["dlblue"])
        self.ax[0].set_xlabel(r"$x_0$") 
        self.ax[0].set_ylabel(r"$x_1$")         
    
    def linear_data(self,redraw=False):
        if not redraw:
            m = 30
            n = 2
            c = 0
            x_train = np.arange(0,m,1)
            np.random.seed(1)
            y_ideal = x_train**2 + c
            y_train = y_ideal + 0.7 * y_ideal*(np.random.sample((m,))-0.5)
            self.x_ideal = x_train #当 X 中加入新数据时用于重绘
            self.X = x_train
            self.y = y_train
            self.y_ideal = y_ideal
        else:
            self.ax[0].set_xlim(self.xlim)
            self.ax[0].set_ylim(self.ylim)

        self.ax[0].scatter(self.X,self.y, label="y")
        self.ax[0].plot(self.x_ideal, self.y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
        self.ax[0].set_title("OverFitting Example: Linear Data Set (quadratic with noise)",fontsize = 14)   
        self.ax[0].set_xlabel("x"); self.ax[0].set_ylabel("y")
        self.ax0ledgend = self.ax[0].legend(loc='lower right')
        self.ax[0].text(0.5,0.93, "Click on plot to add data",
                        fontsize=12, ha='center',transform=self.ax[0].transAxes, color=dlc["dlblue"])
        if not redraw:
            self.xlim = self.ax[0].get_xlim()
            self.ylim = self.ax[0].get_ylim()


    @output.capture()  # 调试
    def add_data(self, event):
        if self.logistic:
            self.add_data_logistic(event)
        else:
            self.add_data_linear(event)

    @output.capture()  # 调试
    def add_data_logistic(self, event):
        if event.inaxes == self.ax[0]:
            x0_coord = event.xdata
            x1_coord = event.ydata
            
            if event.key == None:
                self.ax[0].scatter(x0_coord, x1_coord, marker='x', s=10, c = 'red', label="y=1")
                self.y = np.append(self.y,1)
            else:
                self.ax[0].scatter(x0_coord, x1_coord, marker='o', s=10, label="y=0", facecolors='none',
                                   edgecolors=dlc['dlblue'],lw=3)
                self.y = np.append(self.y,0)
            self.X = np.append(self.X,np.array([[x0_coord, x1_coord]]),axis=0)
        self.fig.canvas.draw()
        
    def add_data_linear(self, event):
        if event.inaxes == self.ax[0]:
            x_coord = event.xdata
            y_coord = event.ydata
            
            self.ax[0].scatter(x_coord, y_coord, marker='o', s=10, facecolors='none',
                                   edgecolors=dlc['dlblue'],lw=3)
            self.y = np.append(self.y,y_coord)
            self.X = np.append(self.X,x_coord)
            self.fig.canvas.draw()

    @output.capture()  # 调试
    def fitdata_clicked(self,event):
        if self.logistic == True:
            self.logistic_regression()
        else:
            self.linear_regression()
        
    def linear_regression(self):
        self.ax[0].clear()
        self.fig.canvas.draw()

        # 用映射后的特征集 mapped_X 创建并拟合模型。
        self.X_mapped, _ =  map_one_feature(self.X, self.degree)
        self.X_mapped_scaled, self.X_mu, self.X_sigma  = zscore_normalize_features(self.X_mapped)
        
        #linear_model = LinearRegression()
        linear_model = Ridge(alpha=self.lambda_, normalize=True, max_iter=10000)
        linear_model.fit(self.X_mapped_scaled, self.y ) 
        self.w = linear_model.coef_.reshape(-1,)
        self.b = linear_model.intercept_
        x = np.linspace(*self.xlim,30)  #绘制与数据无关的直线，该直线会因数据乱序而受影响
        xm, _ =  map_one_feature(x, self.degree)
        xms = (xm - self.X_mu)/ self.X_sigma
        y_pred = linear_model.predict(xms)
        
        #self.fig.canvas.draw()
        self.linear_data(redraw=True)
        self.ax0yfit = self.ax[0].plot(x, y_pred, color = "blue", label="y_fit")
        self.ax0ledgend = self.ax[0].legend(loc='lower right')
        self.fig.canvas.draw()

    def logistic_regression(self):
        self.ax[0].clear()
        self.fig.canvas.draw()

        # 用映射后的特征集 mapped_X 创建并拟合模型。
        self.X_mapped, _ =  map_feature(self.X[:, 0], self.X[:, 1], self.degree)
        self.X_mapped_scaled, self.X_mu, self.X_sigma  = zscore_normalize_features(self.X_mapped)
        if self.regularize == False or self.lambda_ == 0:
            lr = LogisticRegression(penalty='none', max_iter=10000)
        else:
            C = 1/self.lambda_
            lr = LogisticRegression(C=C, max_iter=10000)

        lr.fit(self.X_mapped_scaled,self.y)
        #print(lr.score(self.X_mapped_scaled, self.y))
        self.w = lr.coef_.reshape(-1,)
        self.b = lr.intercept_
        #print(self.w, self.b)
        self.logistic_data(redraw=True)
        self.contour = plot_decision_boundary(self.ax[0],[-1,1],[-1,1], self.y, predict_logistic, self.w, self.b, 
                       scaler=True, mu=self.X_mu, sigma=self.X_sigma, degree=self.degree )
        self.fig.canvas.draw()

    @output.capture()  # 调试
    def update_equation(self, idx, firsttime=False):
        #print(f"Update equation, index = {idx}, firsttime={firsttime}")
        self.degree = idx+1
        if firsttime:
            self.eqtext = []
        else:
            for artist in self.eqtext:
                #print(artist)
                artist.remove()
            self.eqtext = []
        if self.logistic:
            _, equation =  map_feature(self.X[:, 0], self.X[:, 1], self.degree)
            str = 'f_{wb} = sigmoid(' 
        else:
            _, equation =  map_one_feature(self.X, self.degree)
            str = 'f_{wb} = ('
        bz = 10
        seq = equation.split('+')
        blks = math.ceil(len(seq)/bz)
        for i in range(blks):
            if i == 0:
                str = str +  '+'.join(seq[bz*i:bz*i+bz])
            else:
                str = '+'.join(seq[bz*i:bz*i+bz])
            str = str + ')' if i == blks-1 else str + '+'
            ei = self.ax[1].text(0.01,(0.75-i*0.25), f"${str}$",fontsize=9, transform = self.ax[1].transAxes, ma='left', va='top' )
            self.eqtext.append(ei)
        self.fig.canvas.draw()

plt.close("all")
w_in = np.zeros_like(y_train)
b_in = 0.
ofit = overfit_example(X_train, y_train, w_in, b_in,True)
ofit = overfit_example(X_train, y_train, w_in, b_in,False)

plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [32]:
np.linspace(*ofit.xlim,5)

AttributeError: 'overfit_example' object has no attribute 'xlim'

In [ ]:
class equation_manager:
    
    @output.capture()  # 调试
    def __init__(self,ax, logistic=False):
        self.ax = ax
        self.fig = ax.figure
        self.logistic = logistic
        self.init_state = [True, False, False, False, False, False]
        self.axdegree = plt.axes([0.1,0.02,0.15,0.2 ])  #lx,by,w,h
        self.button  = CheckButtons(self.axdegree, ['1','2','3','4','5','6'], self.init_state)
        self.button.on_clicked(self.button_clicked)
        self.degreetxt = self.fig.text(0.1, 0.02+0.21, "Degree", fontsize=12)
        self.status = self.button.get_status()
        #self.update_equation(self.status.index(True)+1, firsttime=True)
        self.button_clicked(None, firsttime=True)
        
    def reinit(self, logistic):
        self.logistic = logistic
        self.button.eventson = False
        self.button.set_active(self.status.index(True))  #关闭旧的
        self.button.eventson = True
        self.button.set_active(self.init_state.index(True))  #打开 init，触发更新
         
    @output.capture()  # 调试
    def button_clicked(self, event, firsttime=False):
        '''firsttime 来自 __init__，而不是按钮按下'''
        new_status = self.button.get_status()
        new = [self.status[i] ^ new_status[i] for i in range(len(self.status))]
        newidx = new.index(True)
        self.button.eventson = False
        self.button.set_active(self.status.index(True))  #关闭旧的；若相同则重新启用
        self.button.eventson = True
        self.status = self.button.get_status()
        self.update_equation(newidx+1,firstttime)
        self.degree = self.status.index(True)+1
   
    @output.capture()  # 调试
    def update_equation(self, degree, firsttime=False):
        if firsttime:
            self.eqtext = []
        else:
            for artist in self.eqtext:
                #print(artist)
                artist.remove()
            self.eqtext = []

        self.X_mapped, equation =  map_feature(X_train[:, 0], X_train[:, 1], degree)
        bz = 10
        seq = equation.split('+')
        blks = math.ceil(len(seq)/bz)
        for i in range(blks):
            if i == 0:
                str = 'f_{wb} = sigmoid('  + '+'.join(seq[bz*i:bz*i+bz])
            else:
                str = '+'.join(seq[bz*i:bz*i+bz])
            str = str + ')' if i == blks-1 else str + '+'
            ei = self.ax.text(0.01,(0.75-i*0.25), f"${str}$",fontsize=9, transform = self.ax.transAxes, ma='left', va='top' )
            self.eqtext.append(ei)
        self.fig.canvas.draw()

    

<a name='FeatureMap'></a>
##  制造过拟合……多项式特征映射
在真实数据集中，“真”与“假”特征之间的边界很少是一条直线。要创建非线性决策边界，我们的模型需要支持非线性特征。具体来说，如果特征集中有两个特征 $x_1$ 和 $x_2$，我们可以构建一个 2 次模型：
$$f_{\\mathbf{w},b} = w_0x_1 + w_1x_2 + w_2x_1^2 + w_3x_1x_2 + w_4x_2^2 + b \\tag{1} $$
为此，我们必须把两特征数据集转换成包含所有特征组合的特征集。上面的 `map_feature` 程序正是为此提供的。

In [12]:
X_tmp = np.array([[2,0],[0,3],[2,3]] )  # 为说明公式而选取的取值
print("Shape before feature mapping:", X_tmp.shape)
print(X_tmp, "\n")

mapped_X, descrip =  map_feature(X_tmp[:, 0], X_tmp[:, 1],degree = 2)

print("Shape after feature mapping:", mapped_X.shape)
print(mapped_X)

Shape before feature mapping: (3, 2)
[[2 0]
 [0 3]
 [2 3]] 

Shape after feature mapping: (3, 5)
[[2 0 4 0 0]
 [0 3 0 0 9]
 [2 3 4 6 9]]


把结果与上面的公式 (1) 对比一下。

当然，我们不必止步于二次。map_features 的 `degree` 参数决定所生成多项式的次数。次数取决于你试图拟合的曲线的复杂程度。增大次数可以让模型拟合更不规则的边界，但也可能导致过拟合。由于包含所有交叉项，特征/参数的数量会呈指数增长。也可以用 Sklearn 的 [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) 来创建特征映射。

让我们把上面的数据集转换成支持 6 次的形式。

In [13]:
X_train = np.array([[2,0],[0,3],[2,3]] )  # 为说明公式而选取的取值
print("Original shape of data:", X_train.shape)
degree = 6
X_mapped, equation =  map_feature(X_train[:, 0], X_train[:, 1], degree)
print(equation)
print("Shape after feature mapping:", X_mapped.shape)
foo=md(f"equation: ${equation}$")
foo2 = f"{equation}"
foo2

Original shape of data: (3, 2)
w_{0}x_0 + w_{1}x_1 + w_{2}x_0^{2} + w_{3}x_0x_1 + w_{4}x_1^{2} + w_{5}x_0^{3} + w_{6}x_0^{2}x_1 + w_{7}x_0x_1^{2} + w_{8}x_1^{3} + w_{9}x_0^{4} + w_{10}x_0^{3}x_1 + w_{11}x_0^{2}x_1^{2} + w_{12}x_0x_1^{3} + w_{13}x_1^{4} + w_{14}x_0^{5} + w_{15}x_0^{4}x_1 + w_{16}x_0^{3}x_1^{2} + w_{17}x_0^{2}x_1^{3} + w_{18}x_0x_1^{4} + w_{19}x_1^{5} + w_{20}x_0^{6} + w_{21}x_0^{5}x_1 + w_{22}x_0^{4}x_1^{2} + w_{23}x_0^{3}x_1^{3} + w_{24}x_0^{2}x_1^{4} + w_{25}x_0x_1^{5} + w_{26}x_1^{6} +  b
Shape after feature mapping: (3, 27)


'w_{0}x_0 + w_{1}x_1 + w_{2}x_0^{2} + w_{3}x_0x_1 + w_{4}x_1^{2} + w_{5}x_0^{3} + w_{6}x_0^{2}x_1 + w_{7}x_0x_1^{2} + w_{8}x_1^{3} + w_{9}x_0^{4} + w_{10}x_0^{3}x_1 + w_{11}x_0^{2}x_1^{2} + w_{12}x_0x_1^{3} + w_{13}x_1^{4} + w_{14}x_0^{5} + w_{15}x_0^{4}x_1 + w_{16}x_0^{3}x_1^{2} + w_{17}x_0^{2}x_1^{3} + w_{18}x_0x_1^{4} + w_{19}x_1^{5} + w_{20}x_0^{6} + w_{21}x_0^{5}x_1 + w_{22}x_0^{4}x_1^{2} + w_{23}x_0^{3}x_1^{3} + w_{24}x_0^{2}x_1^{4} + w_{25}x_0x_1^{5} + w_{26}x_1^{6} +  b'

注意，用 6 次多项式时，我们现在有 27 个特征！
<a name='FitModel'></a>
## 拟合模型

我们将使用之前实验中介绍过的 SkLearn 的 `LogisticRegression` 功能。需要注意一点，这个程序内置了正则化。我们会启用和禁用它，以突出过拟合的各个方面。要禁用它，把命令行参数 `penalty` 设为 `none`。启用时，命令行参数 `C` 控制使用多少正则化。

第一步是对数据做缩放。事实证明，有了二次项后，模型在不使用正则化（我们在这个第一个实验中不使用）的情况下无法拟合，因此我们会对数据做缩放。这类似于第一周介绍的特征缩放/均值归一化。

In [14]:
X_mapped_scaled, X_mu, X_sigma  = zscore_normalize_features(X_mapped)

In [15]:
w_in  = np.zeros_like(X_mapped_scaled[0])
b_in  = 0.
alpha = 0.01
num_iters = 1000000

w_out, b_out, _ = gradient_descent(X_mapped_scaled, y_train, w_in, b_in, alpha, num_iters, logistic=True) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

ValueError: operands could not be broadcast together with shapes (3,1) (50,1) 

In [16]:
#w_out_5M = w_out
#b_out_5M = b_out
from lab_utils_common import compute_cost_matrix, sigmoid
compute_cost_matrix(X_mapped_scaled, y_train, w_out.reshape(-1,1), b_out, logistic=True, lambda_=0)

NameError: name 'w_out' is not defined

现在我们有了训练好的模型，让我们把原始数据（而非预测值）连同从模型得到的决策边界一起画出来。查看上面的 `plot_decision_boundary` 了解具体做法。

In [17]:
#plot_decision_boundary([-1,1],[-1,1], y_train,lr.predict, scaler=scaler )
fig,ax = plt.subplots(1,1, figsize=(4,4))
plot_decision_boundary(ax,[-1,1],[-1,1], y_train, predict_logistic, w_out, 
                       b_out, scaler=True, mu=X_mu, sigma=X_sigma, degree=degree )
plot_data(X_train,y_train,ax,s=10)
ax.set_title(f"Example of overfitting, \ndegree {degree}, no regularization")
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

NameError: name 'w_out' is not defined

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

<center> <img  src="./图片/C1_W3_Lab07_overfitting.PNG" width="440" height="440"/>

> **图中文字（中文）**：过拟合的例子：高次多项式、无正则化   <center/>

In [18]:
# 用映射后的特征集 mapped_X 创建并拟合模型。
lr = LogisticRegression(penalty='none', max_iter=10000)
lr.fit(X_mapped_scaled,y_train)


ValueError: Found input variables with inconsistent numbers of samples: [3, 50]

In [19]:
print(lr.score(X_mapped_scaled, y_train))
w_lr = lr.coef_.reshape(-1,)
b_lr = lr.intercept_
print(w_lr,b_lr)
#plot_decision_boundary([-1,1],[-1,1], y_train,lr.predict, scaler=scaler )
fig,ax = plt.subplots(1,1, figsize=(4,4))
plot_decision_boundary(ax,[-1,1],[-1,1], y_train, predict_logistic, w_lr, 
                       b_lr, scaler=True, mu=X_mu, sigma=X_sigma, degree=degree )
plot_data(X_train,y_train,ax,s=10)
ax.set_title(f"Example of overfitting, \ndegree {degree}, no regularization")
plt.show()

NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [20]:
from lab_utils_common import compute_cost_matrix, sigmoid
compute_cost_matrix(X_mapped_scaled, y_train, w_lr.reshape(-1,1), b_lr, logistic=True, lambda_=0)

NameError: name 'w_lr' is not defined

In [21]:
f_wb.shape

NameError: name 'f_wb' is not defined

In [22]:
f_wb = X_mapped_scaled @ w_lr + b_lr
Blr = -(y_train * f_wb) + np.log(1+np.exp(f_wb))
for i in range(5,5+6):
    print(f_wb[i*6:i*6+6])
    print(y_train[i*6:i*6+6])
    print(Blr[i*6:i*6+6])
    print()

NameError: name 'w_lr' is not defined

哇，模型把数据分得非常漂亮！然而，这大概并不是我们想要的结果。
我们可以用两种办法减轻过拟合：
- 正则化
- 降低多项式的次数。

<a name='ReduceOverfitting'></a>
## 使用正则化减轻过拟合
接下来的实验会更详细地讲正则化，这里我们先简要探索一下。
让我们再次拟合模型，但这次加入正则化。

In [ ]:
# 用映射后的特征集 mapped_X 创建并拟合模型。
lr = LogisticRegression(max_iter=1000, C=1)
lr.fit(mapped_X,y_train)

# 打印拟合效果评估，1 为最好。
print("fitting score:",lr.score(mapped_X, y_train))

In [ ]:
plot_decision_boundary([-1,1],[-1,1], y_train,lr.predict)
plot_data(X_train,y_train)
plt.title("Example of overfitting, degree 6, with regularization, C=1")
plt.show()

加入一些正则化后，决策边界合理多了。
修改上面 `C` 的取值，试试更多或更少的正则化。C 必须严格为正。小于 1 的值使正则化最大化，较大的值则使正则化最小化。

##### 降低多项式的次数
6 次多项式可能超出需要！我们可以降低次数来约束模型。
为此，我们需要重新生成映射后的数据并重新拟合模型。

In [ ]:
print("Original shape of data:", X_train.shape)
degree = 2
mapped_X =  map_feature(X_train[:, 0], X_train[:, 1],degree)

print("Shape after feature mapping:", mapped_X.shape)

In [ ]:
# 用映射后的特征集 mapped_X 创建并拟合模型。
lr = LogisticRegression(penalty='none', max_iter=1000, C=1)
lr.fit(mapped_X,y_train)

# 打印拟合效果评估，1 为最好。
print("fit score:", lr.score(mapped_X, y_train))

In [ ]:
plot_decision_boundary([-1,1],[-1,1], y_train,lr.predict)
plot_data(X_train,y_train)
plt.title("Example of overfitting, degree 2, with no regularization")
plt.show()

还不错！当然，在这个例子里我们事先就知道数据是二次的，也知道 2 次多项式会是个好选择。试着改动上面的 `degree`，看看多项式次数对过拟合的影响。